# Samsung Cloud Firmware → TWRP Device Tree

This notebook now bundles the **full twrpdtgen-NEXT workflow** into the cloud path.

It can either:

1. download/decrypt Samsung firmware in Colab and return only one extracted `.img`, or
2. continue directly into twrpdtgen, generate the complete candidate TWRP device tree, ZIP it, and download only that ZIP to your computer.

The full firmware is processed inside Colab and is removed after extraction by default. Advanced retention settings remain available through the CLI.
All setup and generation commands run in the foreground with live terminal output. Stop the active cell to interrupt the current command.


In [ ]:
#@title 1. Set up or update TDTNext
from pathlib import Path
import os, shutil, subprocess
import codecs, errno, pty, select, signal, sys, time

def foreground(command):
    """Run synchronously; stream real terminal stdout/stderr into this cell."""
    command = [str(item) for item in command]
    print('Running:', ' '.join(command), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1', PIP_PROGRESS_BAR='on',
               TDTNEXT_FOREGROUND='1', TERM='xterm')
    master, slave = pty.openpty()
    process = None
    decoder = codecs.getincrementaldecoder('utf-8')('replace')
    last_output = time.monotonic()
    try:
        try:
            process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
                                       stdout=slave, stderr=slave, env=env,
                                       start_new_session=True)
        finally:
            os.close(slave)
        while True:
            ready, _, _ = select.select([master], [], [], 1)
            if not ready:
                if time.monotonic() - last_output >= 15:
                    print('Still running...', flush=True)
                    last_output = time.monotonic()
                continue
            try:
                chunk = os.read(master, 65536)
            except OSError as exc:
                if exc.errno == errno.EIO:
                    break
                raise
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
            last_output = time.monotonic()
        sys.stdout.write(decoder.decode(b'', final=True))
        sys.stdout.flush()
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    except BaseException:
        if process is not None and process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait()
        raise
    finally:
        os.close(master)
    print('Completed.', flush=True)


repo = Path('/content/twrpdtgen-NEXT')
url = 'https://github.com/RectifyData/twrpdtgen-NEXT.git'
if (repo / '.git').is_dir():
    foreground(['git', '-C', str(repo), 'pull', '--ff-only', '--progress'])
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a Git checkout. Move it aside and rerun.')
else:
    foreground(['git', 'clone', '--progress', url, str(repo)])
os.chdir(repo)

packages = {'cpio':'cpio', 'lz4':'lz4', 'xz':'xz-utils', 'bzip2':'bzip2',
            'gzip':'gzip', 'unzip':'unzip', 'zip':'zip', 'file':'file'}
missing = [package_name for command, package_name in packages.items() if not shutil.which(command)]
if missing:
    foreground(['apt-get', 'update'])
    foreground(['apt-get', 'install', '-y', '--no-install-recommends', *missing])

foreground(['sh', str(repo / 'tdtnext'), '--setup-only'])
next_python = str(repo / '.tdtnext-venv/bin/python')
foreground(['git', 'rev-parse', '--short', 'HEAD'])
foreground([next_python, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'])
print('TDTNext ready.', flush=True)


## Firmware and output options

Choose extraction or tree generation. Leave firmware_version empty for latest FUS firmware. Package auto searches supported Odin packages. Recovery fragment and stock-root inclusion apply only to tree generation. The Samsung cloud path extracts a single recovery-bearing image; use generic Full firmware mode for split-image or vendor/odm analysis.


In [ ]:
#@title 2. Firmware and output options
workflow = 'Generate TWRP device tree' #@param ['Extract image only', 'Generate TWRP device tree']
model = 'SM-A207F' #@param {type:'string'}
region = 'NEE' #@param {type:'string'}
partition = 'recovery' #@param {type:'string'}
firmware_version = '' #@param {type:'string'}
package = 'AP' #@param ['auto', 'AP', 'BL', 'CP', 'CSC', 'HOME_CSC', 'USERDATA']
threads = 8 #@param {type:'integer'}
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot', 'init_boot']
recovery_root_mode = 'auto' #@param ['auto', 'minimal', 'full']
recovery_fragment = '' #@param {type:'string'}
include_stock_root = False #@param {type:'boolean'}


In [ ]:
#@title 3. Run and download
from pathlib import Path
import subprocess, zipfile
from google.colab import files

if not model.strip() or not region.strip() or not partition.strip():
    raise ValueError('Model, region and partition are required.')

if not 1 <= threads <= 32:
    raise ValueError('Threads must be between 1 and 32.')
model, region, partition = model.strip(), region.strip(), partition.strip()
name = Path(partition).name
if name.lower().endswith('.lz4'):
    name = name[:-4]
if not name.lower().endswith(('.img', '.bin')):
    name += '.img'

image_output = Path('/content') / name
tree_output = Path('/content/twrp-tree')
tree_archive = Path('/content/twrp-device-tree.zip')

if workflow == 'Extract image only':
    image_output.unlink(missing_ok=True)
    args = [
        'fetch', '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '-o', str(image_output),
        '--overwrite',
    ]
    if firmware_version.strip():
        args.extend(['-v', firmware_version.strip()])
    cmd = [next_python, '-m', 'twrpdtgen.cloud_slice', *args]
    print('Extracting image:', name)
    foreground(cmd)
    if not image_output.is_file():
        raise FileNotFoundError(f'Extraction finished without {image_output}')
    files.download(str(image_output))

elif workflow == 'Generate TWRP device tree':
    tree_archive.unlink(missing_ok=True)
    args = [
        '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '--tree-output', str(tree_output),
        '--archive', str(tree_archive),
        '--skeleton', skeleton,
        '--recovery-layout', recovery_layout,
        '--recovery-root-mode', recovery_root_mode,
        '--overwrite',
    ]
    if firmware_version.strip():
        args.extend(['-v', firmware_version.strip()])
    if recovery_fragment.strip():
        args.extend(['--recovery-fragment', recovery_fragment.strip()])
    if include_stock_root:
        args.append('--include-stock-root')
    cmd = [next_python, '-m', 'twrpdtgen.cloud_tree', *args]
    print('Generating TWRP tree:', name)
    foreground(cmd)
    if not tree_archive.is_file() or not zipfile.is_zipfile(tree_archive):
        raise FileNotFoundError(f'Generation finished without {tree_archive}')
    files.download(str(tree_archive))

else:
    raise ValueError(f'Unknown workflow: {workflow!r}')


## Results

**Extract image only:** download the selected `.img` file.

**Generate TWRP device tree:** download `twrp-device-tree.zip`. The expanded tree also remains in `/content/twrp-tree/` for inspection until the Colab runtime ends.

The ZIP is a **candidate source tree**, not a boot-verified recovery. Check partition geometry, kernel, AVB, SELinux, encryption and real hardware compatibility before flashing.

The archive excludes `recovery/stock-root/` by default because the original ramdisk is only needed for auditing and can be very large. The expanded stock-root remains in Colab. Symlinks are preserved in the ZIP and require a symlink-capable extractor.

Generation and extraction run in new Python subprocesses, so a Colab kernel won't mix old imported twrpdtgen code with newly pulled sources.